# Segmentation des plans d'eau (Sentinel-2) — entraînement sur Kaggle (GPU)

1. Dans ce notebook Kaggle, clique **Add Input** et ajoute le dataset `franciscoescobar/satellite-images-of-water-bodies`.
2. Dans le panneau de droite, mets **Accelerator = GPU T4 x2** (ou P100).
3. Exécute toutes les cellules (Run All) — ou utilise **Save Version > Save & Run All (Commit)** pour un Output fiable.
4. À la fin, télécharge `/kaggle/working/model.pt` (panneau Output) et dépose-le dans `models/model.pt` du projet local.
5. (Optionnel) Télécharge aussi `/kaggle/working/mlruns/` : c'est le suivi MLflow de ce run (paramètres, métriques par epoch, artefact du modèle) — dépose-le dans `mlruns/` à la racine du projet local pour le visualiser avec `mlflow ui`.

In [ ]:
!pip install -q segmentation-models-pytorch mlflow

In [ ]:
import glob
import os

import numpy as np
import segmentation_models_pytorch as smp
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

INPUT_ROOT = '/kaggle/input'

print('Datasets mounted under', INPUT_ROOT, ':', os.listdir(INPUT_ROOT))


def find_dir(root, name_options):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.lower() in name_options:
                return os.path.join(dirpath, d)
    return None


DATA_ROOT = find_dir(INPUT_ROOT, {'images', 'image'})
MASK_ROOT = find_dir(INPUT_ROOT, {'masks', 'mask'})

if DATA_ROOT is None or MASK_ROOT is None:
    print('Full tree under', INPUT_ROOT, ':')
    for root, dirs, files in os.walk(INPUT_ROOT):
        depth = root[len(INPUT_ROOT):].count(os.sep)
        if depth > 3:
            dirs[:] = []
            continue
        print('  ' * depth + os.path.basename(root) + '/', f'({len(files)} files)' if files else '')
    raise FileNotFoundError(
        'Could not find an Images/Masks folder under ' + INPUT_ROOT +
        '. Check the printed tree above and set DATA_ROOT/MASK_ROOT manually.'
    )

print('Images:', DATA_ROOT, '->', len(os.listdir(DATA_ROOT)), 'files')
print('Masks:', MASK_ROOT, '->', len(os.listdir(MASK_ROOT)), 'files')

In [ ]:
IMAGE_SIZE = 256

image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
mask_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])


class WaterBodiesDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.filenames = sorted(os.listdir(image_dir))
        self.image_dir = image_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        image = Image.open(os.path.join(self.image_dir, name)).convert('RGB')
        mask = Image.open(os.path.join(self.mask_dir, name)).convert('L')
        image = image_transform(image)
        mask = (mask_transform(mask) > 0.5).float()
        return image, mask


full_dataset = WaterBodiesDataset(DATA_ROOT, MASK_ROOT)
val_size = int(0.15 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
print('train:', len(train_ds), 'val:', len(val_ds))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

EPOCHS = 10
LR = 1e-4
BATCH_SIZE = 16

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
).to(device)

loss_fn = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def iou_score(pred, target, eps=1e-6):
    pred = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return ((intersection + eps) / (union + eps)).item()


import mlflow

mlflow.set_tracking_uri('/kaggle/working/mlruns')
mlflow.set_experiment('water-segmentation')
mlflow.start_run()
mlflow.log_params({
    'encoder': 'resnet34',
    'encoder_weights': 'imagenet',
    'image_size': IMAGE_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'batch_size': BATCH_SIZE,
    'loss': 'dice',
    'train_size': len(train_ds),
    'val_size': len(val_ds),
})
print('MLflow run started, tracking at /kaggle/working/mlruns')

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_iou = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            val_iou += iou_score(outputs, masks) * images.size(0)
    val_iou /= len(val_ds)

    mlflow.log_metrics({'train_loss': train_loss, 'val_iou': val_iou}, step=epoch)
    print(f'epoch {epoch + 1}/{EPOCHS}  train_loss={train_loss:.4f}  val_iou={val_iou:.4f}')

In [ ]:
MODEL_PATH = '/kaggle/working/model.pt'
torch.save({
    'state_dict': model.state_dict(),
    'encoder_name': 'resnet34',
    'image_size': IMAGE_SIZE,
}, MODEL_PATH)
print('saved to', MODEL_PATH)

mlflow.log_artifact(MODEL_PATH)
mlflow.end_run()
print('MLflow run logged under /kaggle/working/mlruns (download that folder too if you want to inspect it with `mlflow ui --backend-store-uri mlruns`)')

In [ ]:
import matplotlib.pyplot as plt

model.eval()
images, masks = next(iter(val_loader))
with torch.no_grad():
    preds = torch.sigmoid(model(images.to(device))).cpu()

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
display_images = (images * std + mean).clamp(0, 1)

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i in range(3):
    axes[i, 0].imshow(display_images[i].permute(1, 2, 0))
    axes[i, 0].set_title('image')
    axes[i, 1].imshow(masks[i, 0], cmap='gray')
    axes[i, 1].set_title('masque réel')
    axes[i, 2].imshow(preds[i, 0] > 0.5, cmap='gray')
    axes[i, 2].set_title('prédiction')
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_predictions.png')
plt.show()